In [8]:
import os
import json
import pandas as pd 
import traceback

In [9]:
  import os
  from dotenv import load_dotenv, find_dotenv

In [10]:
from langchain_openai import ChatOpenAI

In [11]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv(find_dotenv())

llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0.3,
)

In [12]:
%pip install -U langchain langchain-openai langchain-core langchain-community pypdf python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [13]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.callbacks import get_openai_callback
from pypdf import PdfReader

C:\Users\GICHUKI THUITA\AppData\Local\Temp\ipykernel_22476\3670598187.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.callbacks import get_openai_callback


In [14]:
import json

RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "3": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
}

In [15]:
TEMPLATE = """
Text:{text}
You are an expert MCQ maker. Given the above text, it is your job to \
create a quiz of {number} multiple choice questions for {subject} students in {tone} tone.
Make sure the questions are not repeated and check all the questions to be conforming to the text as well.
Make sure to format your response like RESPONSE_JSON below and use it as a guide. \
Ensure to make {number} MCQs.
### RESPONSE_JSON
{response_json}
"""

In [16]:
from langchain_core.prompts import PromptTemplate

quiz_generation_prompt = PromptTemplate(
    input_variables=["text", "number", "subject", "tone", "response_json"],
    template=TEMPLATE,
)

In [17]:
from langchain_core.output_parsers import StrOutputParser

quiz_chain = quiz_generation_prompt | llm | StrOutputParser()

In [18]:
TEMPLATE2 = """
You are an expert English grammarian and writer. Given a Multiple Choice Quiz for {subject} students.\
You need to evaluate the complexity of the question and give a complete analysis of the quiz. Only use at max 50 words for complexity analysis.
If the quiz is not at par with the cognitive and analytical abilities of the students,\
update the quiz questions which need to be changed and change the tone such that it perfectly fits the student abilities.
Quiz_MCQs:
{quiz}

Check from an expert English Writer of the above quiz:
"""

In [19]:
from langchain_core.prompts import PromptTemplate

quiz_evaluation_prompt = PromptTemplate(
    input_variables=["subject", "quiz"],
    template=TEMPLATE2,
)

In [20]:
from langchain_core.output_parsers import StrOutputParser

review_chain = quiz_evaluation_prompt | llm | StrOutputParser()

In [21]:
from langchain_core.runnables import RunnablePassthrough

generate_evaluate_chain = (
    RunnablePassthrough.assign(quiz=quiz_chain)
    | RunnablePassthrough.assign(review=review_chain)
)

In [22]:
from langchain_core.runnables import RunnablePassthrough

generate_evaluate_chain = (
    RunnablePassthrough.assign(quiz=quiz_chain)
    | RunnablePassthrough.assign(review=review_chain)
)

In [23]:
from langchain_core.globals import set_verbose

set_verbose(True)

In [24]:
%pip install -U langchain-google-genai

Note: you may need to restart the kernel to use updated packages.


In [25]:
import os
from dotenv import load_dotenv, find_dotenv
from google import genai

load_dotenv(find_dotenv(), override=True)
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

for m in client.models.list():
    if "generateContent" in (m.supported_actions or []):
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/l

In [26]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv(find_dotenv(), override=True)

candidates = [
    "gemini-flash-latest",
    "gemini-flash-lite-latest",
    "gemini-2.5-flash-lite",
    "gemini-2.5-flash",
    "gemini-3-flash-preview",
    "gemini-3.1-flash-lite",
]

working = None
for name in candidates:
    try:
        test = ChatGoogleGenerativeAI(
            model=name,
            google_api_key=os.getenv("GOOGLE_API_KEY"),
            temperature=0.3,
        )
        reply = test.invoke("Say hello in one short sentence.").content
        print(f"WORKS  {name}: {reply}")
        working = working or name
    except Exception as e:
        print(f"FAILS  {name}: {str(e)[:150]}")

print("\nUse this model:", working)

FAILS  gemini-flash-latest: Error calling model 'gemini-flash-latest' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current 
WORKS  gemini-flash-lite-latest: [{'type': 'text', 'text': 'Hello, how can I help you today?', 'extras': {'signature': 'EmAKXgFpFH0TmkXb8C/fXCayb1HQ3ZAPSB5t6LP2zIg6fIdNepzubnDwD3aV6D620WNwOnGzNMOdB22JkhNhdCzKXy4LiUvhBpx9eoEAPQVywqZXDgYIbYhBD9W7cCyq0Lc='}}]
FAILS  gemini-2.5-flash-lite: Error calling model 'gemini-2.5-flash-lite' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash-lite is
FAILS  gemini-2.5-flash: Error calling model 'gemini-2.5-flash' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer
WORKS  gemini-3-flash-preview: [{'type': 'text', 'text': 'Hello, how can I help you today?', 'extras': {'signature': 'EpwHCpkHAWkUfRP9IDOC35Bp0+7UCzKKC7j+l68EuGtElMiaDzNNcGpBERWzlGhCsOpHQWhtJb/HPg+SwYmcqifgd

In [27]:
import os
import json
import re
from dotenv import load_dotenv, find_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

load_dotenv(find_dotenv(), override=True)

llm = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0.3,
    max_retries=5,
)
print(llm.invoke("Say hello in one short sentence.").text)

Hello, how can I help you today?


In [28]:
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"},
        "correct": "correct answer",
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"},
        "correct": "correct answer",
    },
    "3": {
        "mcq": "multiple choice question",
        "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"},
        "correct": "correct answer",
    },
}

In [29]:
TEMPLATE = """
Text:{text}
You are an expert MCQ maker. Given the above text, it is your job to \
create a quiz of {number} multiple choice questions for {subject} students in {tone} tone.
Make sure the questions are not repeated and check all the questions to be conforming to the text as well.
Make sure to format your response like RESPONSE_JSON below and use it as a guide. \
Ensure to make {number} MCQs.
Return ONLY valid JSON with no extra text.
### RESPONSE_JSON
{response_json}
"""

TEMPLATE2 = """
You are an expert English grammarian and writer. Given a Multiple Choice Quiz for {subject} students.\
You need to evaluate the complexity of the question and give a complete analysis of the quiz. Only use at max 50 words for complexity analysis.
If the quiz is not at par with the cognitive and analytical abilities of the students,\
update the quiz questions which need to be changed and change the tone such that it perfectly fits the student abilities.
Quiz_MCQs:
{quiz}

Check from an expert English Writer of the above quiz:
"""

quiz_generation_prompt = PromptTemplate(
    input_variables=["text", "number", "subject", "tone", "response_json"],
    template=TEMPLATE,
)

quiz_evaluation_prompt = PromptTemplate(
    input_variables=["subject", "quiz"],
    template=TEMPLATE2,
)

In [30]:
quiz_chain = quiz_generation_prompt | llm | StrOutputParser()
review_chain = quiz_evaluation_prompt | llm | StrOutputParser()

generate_evaluate_chain = (
    RunnablePassthrough.assign(quiz=quiz_chain)
    | RunnablePassthrough.assign(review=review_chain)
)

In [31]:
TEXT = """Artificial Intelligence (AI) is a branch of computer science that aims to create software or machines that exhibit human-like intelligence. This can include learning from experience, understanding natural language, and solving problems."""

result = generate_evaluate_chain.invoke({
    "text": TEXT,
    "number": 5,
    "subject": "pretrained transformer",
    "tone": "simple",
    "response_json": json.dumps(RESPONSE_JSON),
})

print(result["quiz"])
print("\n--- REVIEW ---\n")
print(result["review"])

{
  "1": {
    "mcq": "What is Artificial Intelligence (AI) a branch of?",
    "options": {
      "a": "Computer science",
      "b": "Mechanical engineering",
      "c": "Biology",
      "d": "Mathematics"
    },
    "correct": "a"
  },
  "2": {
    "mcq": "What is the main goal of AI?",
    "options": {
      "a": "To build faster computer processors",
      "b": "To create software or machines that exhibit human-like intelligence",
      "c": "To replace all human jobs",
      "d": "To design video games"
    },
    "correct": "b"
  },
  "3": {
    "mcq": "Which of the following is included as a capability of AI in the text?",
    "options": {
      "a": "Flying airplanes",
      "b": "Cooking food",
      "c": "Learning from experience",
      "d": "Creating physical energy"
    },
    "correct": "c"
  },
  "4": {
    "mcq": "According to the text, what kind of language can AI understand?",
    "options": {
      "a": "Programming language only",
      "b": "Natural language",
    

In [32]:
%pip install -q pandas

Note: you may need to restart the kernel to use updated packages.


In [33]:
import pandas as pd

def clean_json(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return text

quiz_dict = json.loads(clean_json(result["quiz"]))

rows = []
for number, q in quiz_dict.items():
    options = " | ".join(f"{k}: {v}" for k, v in q["options"].items())
    rows.append({"No": number, "MCQ": q["mcq"], "Choices": options, "Correct": q["correct"]})

quiz_table = pd.DataFrame(rows)
quiz_table

,No,MCQ,Choices,Correct
0,1,What is Artificial Intelligence (AI) a branch of?,a: Computer science | b: Mechanical engineerin...,a
1,2,What is the main goal of AI?,a: To build faster computer processors | b: To...,b
2,3,Which of the following is included as a capabi...,a: Flying airplanes | b: Cooking food | c: Lea...,c
3,4,"According to the text, what kind of language c...",a: Programming language only | b: Natural lang...,b
4,5,"Besides learning and understanding language, w...",a: Solving problems | b: Traveling the world |...,a


In [34]:
quiz_table.to_csv("quiz.csv", index=False)

In [35]:
from pypdf import PdfReader

def read_file(path):
    if path.lower().endswith(".pdf"):
        reader = PdfReader(path)
        return "\n".join(page.extract_text() or "" for page in reader.pages)
    elif path.lower().endswith(".txt"):
        with open(path, "r", encoding="utf-8") as f:
            return f.read()
    else:
        raise ValueError("Unsupported file type. Use a .pdf or .txt file.")

In [36]:
file_path = "data.txt"
TEXT = read_file(file_path)

print("Characters read:", len(TEXT))
print(TEXT[:500])

Characters read: 6123
HOW A TRANSFORMER WORKS IN AN AI SETUP
Study Notes for AI Engineering and Generative AI
1. Introduction
In Artificial Intelligence, a Transformer is a neural network architecture that helps AI systems learn relationships between pieces of data, especially words and other tokens in a sequence. Transformers power many modern Large Language Models (LLMs), including GPT-style models. They are used in chatbots, AI agents, document analysis, code assistants, translation and many other applications.
2.


In [37]:
print(TEXT)

HOW A TRANSFORMER WORKS IN AN AI SETUP
Study Notes for AI Engineering and Generative AI
1. Introduction
In Artificial Intelligence, a Transformer is a neural network architecture that helps AI systems learn relationships between pieces of data, especially words and other tokens in a sequence. Transformers power many modern Large Language Models (LLMs), including GPT-style models. They are used in chatbots, AI agents, document analysis, code assistants, translation and many other applications.
2. The Basic Idea
Consider the sentence: “The electrician repaired the circuit because it was faulty.” To interpret the word “it”, a model needs to relate it to other words in the sentence. A Transformer uses a mechanism called self-attention to learn which parts of the input are relevant to one another.
Input text: The sentence or other data supplied to the model.
Tokenization: Split the input into tokens such as words, word parts or punctuation.
Embeddings and positional information: Represen

In [38]:
from langchain_core.callbacks import get_usage_metadata_callback

In [39]:
TEXT = read_file("data.txt")
NUMBER = 5
SUBJECT = "Electrical Installation"   # change to your subject
TONE = "simple"

print("Characters read:", len(TEXT))

Characters read: 6123


In [40]:
with get_usage_metadata_callback() as cb:
    response = generate_evaluate_chain.invoke({
        "text": TEXT,
        "number": NUMBER,
        "subject": SUBJECT,
        "tone": TONE,
        "response_json": json.dumps(RESPONSE_JSON),
    })

print(response["quiz"])
print("\n--- REVIEW ---\n")
print(response["review"])
print("\n--- TOKENS ---")
print(cb.usage_metadata)

{
  "1": {
    "mcq": "In the sentence 'The electrician repaired the circuit because it was faulty,' what mechanism does a Transformer use to understand what 'it' refers to?",
    "options": {
      "a": "Tokenization",
      "b": "Self-attention",
      "c": "Positional encoding",
      "d": "Feed-forward networking"
    },
    "correct": "b"
  },
  "2": {
    "mcq": "During which stage of a Transformer is the input text split into smaller units like whole words, word parts, or punctuation marks?",
    "options": {
      "a": "Tokenization",
      "b": "Self-attention",
      "c": "Output generation",
      "d": "Embedding"
    },
    "correct": "a"
  },
  "3": {
    "mcq": "In the library analogy used for Query, Key, and Value (Q, K, V), what does the Query represent?",
    "options": {
      "a": "The useful content of the selected book",
      "b": "A book's description",
      "c": "Your search question",
      "d": "The library building"
    },
    "correct": "c"
  },
  "4": {
  

In [43]:
def find_quiz(data):
    """Dig through the response until we find the {"1": {"mcq":..., "options":..., "correct":...}} structure."""
    # LangChain message object
    if hasattr(data, "content"):
        data = data.content

    # List of content blocks, e.g. [{"type": "text", "text": "..."}]
    if isinstance(data, list):
        data = "".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in data)

    # JSON text, possibly wrapped in ```json fences
    if isinstance(data, str):
        cleaned = re.sub(r"^```(?:json)?|```$", "", data.strip(), flags=re.MULTILINE).strip()
        data = json.loads(cleaned)

    if isinstance(data, dict):
        # Found it: values are question dicts
        first = next(iter(data.values()), None)
        if isinstance(first, dict) and "mcq" in first:
            return data
        # Otherwise it's a wrapper: search inside each value
        for v in data.values():
            try:
                return find_quiz(v)
            except Exception:
                continue

    raise ValueError("Could not find quiz questions in response")


quiz = find_quiz(response)

quiz_table_data = []
for key, value in quiz.items():
    mcq = value["mcq"]
    options = " | ".join(f"{k}: {v}" for k, v in value["options"].items())
    correct = value["correct"]
    quiz_table_data.append({"MCQ": mcq, "Choices": options, "Correct": correct})

quiz_df = pd.DataFrame(quiz_table_data)
quiz_df.to_csv("machinelearning.csv", index=False)

quiz_df

,MCQ,Choices,Correct
0,multiple choice question,a: choice here | b: choice here | c: choice he...,correct answer
1,multiple choice question,a: choice here | b: choice here | c: choice he...,correct answer
2,multiple choice question,a: choice here | b: choice here | c: choice he...,correct answer


In [44]:
def parse_quiz(data):
    # Message object or list of content blocks -> plain text
    if hasattr(data, "content"):
        data = data.content
    if isinstance(data, list):
        data = "".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in data)

    # Text -> dict (take everything from the first { to the last })
    if isinstance(data, str):
        start, end = data.find("{"), data.rfind("}")
        data = json.loads(data[start:end + 1])
    return data


# Use the "quiz" key from the chain output, not "response_json" (that's just the template)
raw_quiz = response["quiz"] if isinstance(response, dict) and "quiz" in response else response
quiz = parse_quiz(raw_quiz)

quiz_table_data = []
for key, value in quiz.items():
    mcq = value["mcq"]
    options = " | ".join(f"{k}: {v}" for k, v in value["options"].items())
    correct = value["correct"]
    quiz_table_data.append({"MCQ": mcq, "Choices": options, "Correct": correct})

quiz_df = pd.DataFrame(quiz_table_data)
quiz_df.to_csv("machinelearning.csv", index=False)

quiz_df

,MCQ,Choices,Correct
0,In the sentence 'The electrician repaired the ...,a: Tokenization | b: Self-attention | c: Posit...,b
1,During which stage of a Transformer is the inp...,a: Tokenization | b: Self-attention | c: Outpu...,a
2,"In the library analogy used for Query, Key, an...",a: The useful content of the selected book | b...,c
3,Which type of Transformer architecture is typi...,a: Encoder-only | b: Decoder-only | c: Encoder...,b
4,In an AI system designed to help electrical tr...,a: Provide the user interface and buttons | b:...,c


In [1]:
   import sys, os
   sys.path.append(os.path.abspath(".."))

   from src.mcqgenerator.logger import logging

   logging.info("Starting MCQ generation")

In [2]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.mcqgenerator.logger import logging

logging.info("Starting MCQ generation")
print("Logger works")

Logger works


In [3]:
from datetime import datetime

datetime.now().strftime('%m_%d_%Y_%H_%M_%S')

'10_16_2026_11_20_33'